<a href="https://colab.research.google.com/github/Shineii86/LeechBot/blob/main/notebooks/LeechBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
<img src="https://capsule-render.vercel.app/api?type=waving&height=300&color=gradient&text=𝗟𝗲𝗲𝗰𝗵%20𝗕𝗼𝘁&fontAlignY=30&fontSize=100&desc=𝖠𝖽𝗏𝖺𝗇𝖼𝖾𝖽%20𝖳𝖾𝗅𝖾𝗀𝗋𝖺𝗆%20𝖥𝗂𝗅𝖾%20𝖳𝗋𝖺𝗇𝗌𝗅𝗈𝖺𝖽𝖾𝗋&descSize=30" />

**A powerful Pyrogram-based bot to transfer files to Telegram & Google Drive**

![Version](https://img.shields.io/badge/Version-3.1.39-8B5CF6?style=for-the-badge)
![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=for-the-badge&logo=python&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-06B6D4?style=for-the-badge)

---

### ✨ Features

| 📥 Download From | 📤 Upload To | 🛠️ Tools |
|:---:|:---:|:---:|
| YouTube, Facebook, Instagram | Telegram | Video Converter (GPU) |
| Google Drive, Mega, Terabox | Google Drive | Archive Handler |
| Pixeldrain, Mediafire, Direct | Directory Leech | Smart Splitting |
| 2000+ sites via yt-dlp | Batch Photos | Download Queue |

---

### 🚀 Quick Start

1. **Fill credentials** in Cell 2 (or use Colab Secrets)
2. Click **Runtime → Run all** or press **Ctrl+F9**
3. Bot starts automatically — send `/start` on Telegram

---

### 📋 Cells

| # | Cell | Purpose |
|:--|:-----|:--------|
| 1 | ♻️ Google Drive Auth | Optional: mount Drive / generate token |
| 2 | 🚀 LeechBot Deployer | Setup + deploy + keep-alive |

</div>

In [ ]:
# @title ♻️ Google Drive Auth
#@markdown <br><center><img src='https://user-images.githubusercontent.com/125879861/255377947-6ac19c35-dbbd-4a9b-bc0e-c603de81c533.png' height="70" alt="Gdrive-logo"/></center>
#@markdown <center><h4><font color=orange><b>Secure Google Drive Integration</b></font></h4></center><br>

from google.colab import auth, drive
from google.auth.transport.requests import Request
import google.auth
import pickle
import os
import time
from IPython.display import clear_output, display, Markdown
import ipywidgets as widgets

# ─────────────────────────────────────────────────────────────
# 🎛️ Configuration Widget
# ─────────────────────────────────────────────────────────────
MODE = widgets.Dropdown(
    options=['Mount Drive', 'Unmount Drive', 'Generate Token', 'Nothing'],
    value='Mount Drive',
    description='🔧 Action:',
    style={'description_width': 'initial'}
)

MOUNT_PATH = widgets.Text(
    value='/content/drive',
    description='📁 Mount Path:',
    disabled=False,
    style={'description_width': 'initial'}
)

TOKEN_PATH = widgets.Text(
    value='/content/token.pickle',
    description='🔑 Token Save Path:',
    disabled=False,
    style={'description_width': 'initial'}
)

USE_COLAB_SECRETS = widgets.Checkbox(
    value=True,
    description='🔐 Use Colab Secrets for credentials (recommended)',
    indent=False
)

config_box = widgets.VBox([MODE, MOUNT_PATH, TOKEN_PATH, USE_COLAB_SECRETS])
display(config_box)

# ─────────────────────────────────────────────────────────────
# 🛠️ Helper Functions
# ─────────────────────────────────────────────────────────────
def print_status(emoji: str, message: str, level: str = "info"):
    """Unified status printer with color coding."""
    colors = {"info": "cyan", "success": "green", "error": "red", "warning": "orange"}
    color = colors.get(level, "white")
    display(Markdown(f"<font color={color}>[{emoji}] {message}</font>"))

def mount_drive(path: str = "/content/drive", force: bool = True):
    """Mount Google Drive with retry logic."""
    for attempt in range(3):
        try:
            print_status("🔗", f"Mounting Drive to {path}... (attempt {attempt+1}/3)")
            drive.mount(path, force_remount=force)
            print_status("✅", "Google Drive mounted successfully!", "success")
            return True
        except Exception as e:
            print_status("⚠️", f"Mount attempt {attempt+1} failed: {e}", "warning")
            time.sleep(2)
    print_status("❌", "Failed to mount Drive after 3 attempts", "error")
    return False

def unmount_drive():
    """Safely unmount Google Drive."""
    try:
        drive.flush_and_unmount()
        print_status("🔓", "Google Drive unmounted successfully!", "success")
        return True
    except Exception as e:
        print_status("⚠️", f"Unmount warning: {e}", "warning")
        return False

def generate_token(save_path: str = "/content/token.pickle"):
    """Generate & save token using Colab authentication."""
    try:
        auth.authenticate_user()
        print_status("🔐", "User authenticated via Colab", "success")

        credentials, project = google.auth.default()

        if credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
            print_status("🔄", "Credentials refreshed", "info")

        with open(save_path, 'wb') as f:
            pickle.dump(credentials, f)

        os.chmod(save_path, 0o600)
        print_status("💾", f"Token saved securely to {save_path}", "success")
        return True

    except Exception as e:
        print_status("❌", f"Token generation failed: {e}", "error")
        return False

# ─────────────────────────────────────────────────────────────
# 🚀 Main Execution
# ─────────────────────────────────────────────────────────────
def execute():
    clear_output(wait=True)
    display(Markdown("### ⚙️ Processing Request...\n"))

    action = MODE.value
    mount_to = MOUNT_PATH.value
    token_to = TOKEN_PATH.value
    use_secrets = USE_COLAB_SECRETS.value

    if action == "Mount Drive":
        mount_drive(mount_to)
    elif action == "Unmount Drive":
        unmount_drive()

    if action == "Generate Token":
        generate_token(token_to)

    if action == "Mount Drive" and use_secrets:
        generate_token(token_to)

    display(Markdown("\n---\n### ✅ Operation Complete!\n<sub>Tip: Store tokens in Colab Secrets for production use.</sub>"))

execute_button = widgets.Button(description="▶️ Execute", button_style="success", icon="check")
execute_button.on_click(lambda b: execute())
display(execute_button)


In [ ]:
# @title **🚀 LeechBot Colab Deployer**
#@markdown <div align="center">
#@markdown   <img src="https://user-images.githubusercontent.com/125879861/255391401-371f3a64-732d-4954-ac0f-4f093a6605e1.png" width="600px">
#@markdown </div>
#@markdown
#@markdown **✨ Features**: Secrets Support • Auto-Recovery • GPU Optimization • Health Checks
#@markdown
#@markdown ---
#@markdown ## 🔐 **Credentials**
#@markdown
#@markdown | Field | Secret Key Name | Required |
#@markdown |-------|----------------|----------|
#@markdown | API_ID | `LEECHBOT_API_ID` | ✅ |
#@markdown | API_HASH | `LEECHBOT_API_HASH` | ✅ |
#@markdown | BOT_TOKEN | `LEECHBOT_BOT_TOKEN` | ✅ |
#@markdown | USER_ID | `LEECHBOT_USER_ID` | ✅ |
#@markdown | DUMP_ID | `LEECHBOT_DUMP_ID` | ✅ |

API_ID = 0  # @param {type:"integer"}
API_HASH = ""  # @param {type:"string"}
BOT_TOKEN = ""  # @param {type:"string"}
USER_ID = 0  # @param {type:"integer"}
DUMP_ID = 0  # @param {type:"integer"}

#@markdown ---
#@markdown ## ⚙️ **Deployment Options**
MOUNT_DRIVE = False  # @param {type:"boolean"}
USE_GPU = True       # @param {type:"boolean"}
ENABLE_LOGS = True   # @param {type:"boolean"}
AUTO_RESTART = True  # @param {type:"boolean"}
REPO_BRANCH = "main"  # @param ["main"]

#@markdown ---
#@markdown > 💡 **Tip**: Click **Runtime → Run all** or press **Ctrl+F9** after filling credentials.

# =============================================================================
#  📦 Imports & Setup
# =============================================================================
import subprocess, sys, os, json, time, shutil, signal
from pathlib import Path
from IPython.display import clear_output, display, Markdown
from google.colab import drive
import logging

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["ALSA_CONFIG_PATH"] = "/dev/null"

logging.basicConfig(
    level=logging.INFO if ENABLE_LOGS else logging.WARNING,
    format='[%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("LeechBot")

# =============================================================================
#  🎨 UI Components
# =============================================================================
class ColabUI:
    @staticmethod
    def banner():
        return """
╔═════════════════════════════════════════╗
║                                          ║
║   🚀 LeechBot - Telegram File Transloader ║
║                                          ║
╠═════════════════════════════════════════╣
║  👤 Dev: Shinei Nouzen                   ║
║  📂 GitHub: Shineii86/LeechBot           ║
║  💬 Telegram: @Shineii86                 ║
╚═════════════════════════════════════════╝
"""

    @staticmethod
    def status(emoji: str, msg: str, level: str = "info"):
        colors = {"info": "#2196F3", "success": "#4CAF50", "error": "#F44336", "warning": "#FF9800"}
        display(Markdown(f'<font color="{colors.get(level, "#999")}">**{emoji} {msg}**</font>'))

# =============================================================================
#  🔐 Credential Management
# =============================================================================
def get_credentials():
    creds = {}
    try:
        from google.colab import userdata
        secrets_map = {
            'API_ID': 'LEECHBOT_API_ID',
            'API_HASH': 'LEECHBOT_API_HASH',
            'BOT_TOKEN': 'LEECHBOT_BOT_TOKEN',
            'USER_ID': 'LEECHBOT_USER_ID',
            'DUMP_ID': 'LEECHBOT_DUMP_ID'
        }
        for key, secret_name in secrets_map.items():
            try:
                value = userdata.get(secret_name)
                creds[key] = int(value) if key in ['API_ID', 'USER_ID', 'DUMP_ID'] else value
                ColabUI.status("🔐", f"Loaded {key} from Colab Secrets", "success")
            except Exception:
                creds[key] = None
    except ImportError:
        ColabUI.status("ℹ️", "Colab Secrets not available; using manual inputs", "info")

    fallbacks = {
        'API_ID': API_ID, 'API_HASH': API_HASH, 'BOT_TOKEN': BOT_TOKEN,
        'USER_ID': USER_ID, 'DUMP_ID': DUMP_ID
    }
    for key in creds:
        if creds[key] is None:
            creds[key] = fallbacks.get(key)
    return creds

def validate_credentials(creds: dict) -> bool:
    required = ['API_ID', 'API_HASH', 'BOT_TOKEN', 'USER_ID', 'DUMP_ID']
    missing = [k for k in required if not creds.get(k)]
    if missing:
        ColabUI.status("❌", f"Missing credentials: {', '.join(missing)}", "error")
        return False
    dump_str = str(creds['DUMP_ID'])
    if len(dump_str) == 10 and not dump_str.startswith('-100'):
        creds['DUMP_ID'] = int("-100" + dump_str)
        ColabUI.status("🔄", "Auto-formatted DUMP_ID", "info")
    return True

# =============================================================================
#  🛠️ Setup Functions
# =============================================================================
def run_command(cmd: str, description: str, retries: int = 3) -> bool:
    for attempt in range(retries):
        try:
            ColabUI.status("⏳", f"{description} (attempt {attempt+1}/{retries})")
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True, timeout=300)
            ColabUI.status("✅", f"{description} completed", "success")
            return True
        except subprocess.CalledProcessError as e:
            logger.warning(f"Command failed: {e.stderr[:200]}")
            if attempt == retries - 1:
                ColabUI.status("❌", f"{description} failed after {retries} attempts", "error")
                return False
            time.sleep(2 ** attempt)
        except subprocess.TimeoutExpired:
            ColabUI.status("⚠️", f"{description} timed out", "warning")
            if attempt == retries - 1:
                return False
    return False

def clone_repo(branch: str = "main") -> bool:
    repo_url = "https://github.com/Shineii86/LeechBot.git"
    target = "/content/leechbot"
    os.chdir("/content")
    if os.path.exists(target):
        shutil.rmtree(target)
        ColabUI.status("🧹", "Cleaned previous installation")
    return run_command(
        f"git clone -b {branch} --depth 1 {repo_url} {target}",
        f"📥 Cloning repo (branch: {branch})"
    )

def install_dependencies() -> bool:
    if not run_command(
        "apt-get update -qq && apt-get install -y -qq ffmpeg aria2 megatools p7zip-full unzip",
        "🔧 Installing system packages"
    ):
        return False
    if not run_command(
        "pip3 install -q --no-cache-dir -r /content/leechbot/requirements.txt",
        "🐍 Installing Python dependencies"
    ):
        return False
    
# =============================================================================
#  📌 Read bot version from cloned repo
# =============================================================================
try:
    from pathlib import Path
    import re
    _cfg = Path("/content/leechbot/config.py").read_text()
    _m = re.search(r'VERSION\s*=\s*["\']([^"\']+)["\']', _cfg)
    BOT_VERSION = _m.group(1) if _m else "unknown"
    ColabUI.status("📦", f"LeechBot version: **{BOT_VERSION}**", "info")
except Exception as e:
    BOT_VERSION = "unknown"
    ColabUI.status("⚠️", f"Could not read version: {e}", "warning")

# Install libtorrent (optional — apt first, conda fallback)
    _lt = run_command("apt-get install -y -qq python3-libtorrent", "🧲 libtorrent (apt)", retries=1)
    if not _lt:
        run_command("conda install -y -q -c conda-forge libtorrent 2>/dev/null", "🧲 libtorrent (conda)", retries=1)
    return True

def check_gpu() -> dict:
    info = {"available": False}
    if not USE_GPU:
        ColabUI.status("ℹ️", "GPU usage disabled by user", "info")
        return info
    try:
        result = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                              shell=True, capture_output=True, text=True, check=True)
        lines = result.stdout.strip().split('\n')
        if lines:
            name, memory = lines[0].split(', ')
            info.update({"available": True, "name": name.strip(), "memory": memory.strip()})
            ColabUI.status("🎮", f"GPU detected: {name} ({memory} VRAM)", "success")
    except Exception:
        try:
            import GPUtil
            gpus = GPUtil.getAvailable(order='memory', limit=1)
            if gpus:
                info["available"] = True
                ColabUI.status("🎮", "GPU acceleration enabled", "success")
        except ImportError:
            pass
    if not info["available"]:
        ColabUI.status("ℹ️", "Using CPU fallback (no GPU detected)", "info")
    return info

def save_config(creds: dict, path: str = "/content/leechbot/credentials.json"):
    with open(path, 'w') as f:
        json.dump(creds, f, indent=2)
    os.chmod(path, 0o600)
    ColabUI.status("💾", "Configuration saved securely", "success")

# =============================================================================
#  🚀 Main Deployment
# =============================================================================
def deploy():
    clear_output(wait=True)
    print(ColabUI.banner())

    # ── Credentials ────────────────────────────────────────
    ColabUI.status("🔐", "Loading credentials...")
    creds = get_credentials()
    if not validate_credentials(creds):
        ColabUI.status("❌", "Deployment aborted: invalid credentials", "error")
        return

    # ── Clone ──────────────────────────────────────────────
    if not clone_repo(REPO_BRANCH):
        return

    # ── Dependencies ───────────────────────────────────────
    if not install_dependencies():
        return

    # ── GPU ────────────────────────────────────────────────
    check_gpu()

    # ── Save Config ────────────────────────────────────────
    save_config(creds)
    os.environ["API_ID"] = str(creds["API_ID"])
    os.environ["API_HASH"] = str(creds["API_HASH"])
    os.environ["BOT_TOKEN"] = str(creds["BOT_TOKEN"])
    os.environ["OWNER_ID"] = str(creds["USER_ID"])
    os.environ["DUMP_ID"] = str(creds["DUMP_ID"])
    ColabUI.status("✅", "Environment variables set", "success")

    # ── Mount Drive ────────────────────────────────────────
    if MOUNT_DRIVE:
        try:
            drive.mount('/content/drive')
            ColabUI.status("☁️", "Google Drive mounted", "success")
        except Exception as e:
            ColabUI.status("⚠️", f"Drive mount failed: {e}", "warning")

    # ── Clean Sessions ─────────────────────────────────────
    for sf in ["/content/leechbot/leechbot_session.session",
               "/content/leechbot/leechbot_session.session-journal"]:
        if os.path.exists(sf):
            os.remove(sf)

    # ── Health Check ───────────────────────────────────────
    v = sys.version.split()[0]
    ok = tuple(int(x) for x in v.split('.')) >= (3, 10)
    ColabUI.status("✅" if ok else "❌", f"Python {v}", "success" if ok else "error")
    for tool, pkg in [("ffmpeg", "ffmpeg"), ("aria2c", "aria2"), ("megadl", "megatools")]:
        try:
            subprocess.run(f"{tool} --version", shell=True, capture_output=True, check=True)
            ColabUI.status("✅", tool, "success")
        except Exception:
            ColabUI.status("❌", f"{tool} — apt install {pkg}", "error")
    try:
        import libtorrent
        ColabUI.status("✅", "libtorrent", "success")
    except Exception:
        ColabUI.status("⚠️", "libtorrent — optional (torrent support)", "warning")
    try:
        import yt_dlp
        ColabUI.status("✅", "yt-dlp", "success")
    except Exception:
        ColabUI.status("❌", "yt-dlp — pip install yt-dlp", "error")

    # ── Summary ────────────────────────────────────────────
    clear_output(wait=True)
    print(ColabUI.banner())
    display(Markdown("""
### ✅ **Deployment Successful!** 🎉

| Command | Description |
|---------|-------------|
| `/start` | Initialize bot & show menu |
| `/tupload` | Upload files to Telegram |
| `/gdupload` | Mirror to Google Drive |
| `/ytupload` | Download via yt-dlp |
| `/settings` | Configure preferences |
| `/help` | Show all available commands |

> ⚠️ **Keep this tab open** while the bot runs.
"""))

    # ── Launch Bot ─────────────────────────────────────────
    # This blocks the cell (which keeps Colab alive)
    ColabUI.status("🚀", "Starting LeechBot...", "success")

    def signal_handler(sig, frame):
        ColabUI.status("🛑", "Received shutdown signal, cleaning up...")
        sys.exit(0)

    if AUTO_RESTART:
        signal.signal(signal.SIGTERM, signal_handler)
        signal.signal(signal.SIGINT, signal_handler)

    os.chdir("/content/leechbot")
    get_ipython().system('python3 -m leechbot')

# ─────────────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────────────
try:
    deploy()
except KeyboardInterrupt:
    ColabUI.status("👋", "Deployment cancelled by user", "warning")
except Exception as e:
    ColabUI.status("💥", f"Unexpected error: {e}", "error")
    if ENABLE_LOGS:
        logger.exception("Full traceback:")
